Install Required Packages

In [ ]:
%%capture
!pip install openpyxl python-docx plotly kaleido pandas numpy matplotlib seaborn

Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime
from google.colab import files
import io
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


Upload Your Files

In [ ]:
print("📁 Select your Excel/CSV files for analysis")
print("You can select multiple files at once!\n")

uploaded = files.upload()

print(f"\n✅ {len(uploaded)} file(s) uploaded successfully!")
for filename in uploaded.keys():
    print(f"  - {filename}")

📁 Select your Excel/CSV files for analysis
You can select multiple files at once!



Saving Activités par Unité et Saison.xlsx to Activités par Unité et Saison.xlsx
Saving Activites_Generales.xlsx to Activites_Generales.xlsx
Saving american-bsa_merit_badges (1).xlsx to american-bsa_merit_badges (1).xlsx
Saving american-bsa_merit_badges.xlsx to american-bsa_merit_badges.xlsx
Saving american-scout-partners (1).xlsx to american-scout-partners (1).xlsx
Saving american-scout-partners.xlsx to american-scout-partners.xlsx
Saving Budgets par Type d'Activité.xlsx to Budgets par Type d'Activité.xlsx
Saving Budgets_et_Finances.xlsx to Budgets_et_Finances.xlsx
Saving Camps_Detailles.xlsx to Camps_Detailles.xlsx
Saving Classeur1.xlsx to Classeur1.xlsx
Saving Détails des Coûts de Camp (Dinars).xlsx to Détails des Coûts de Camp (Dinars).xlsx
Saving event-france_final (1).xlsx to event-france_final (1).xlsx
Saving event-spain (1).xlsx to event-spain (1).xlsx
Saving Membres par Unité et Saison.xlsx to Membres par Unité et Saison.xlsx
Saving tunisian-scouts-blog-posts (1).xlsx to tunisi

 Load and Process Files

In [5]:
def read_file(filename, file_content):
    """Read file handling different formats and encodings"""
    try:
        if filename.endswith('.csv'):
            # Try different encodings for CSV
            for encoding in ['utf-8', 'latin1', 'cp1252']:
                try:
                    df = pd.read_csv(io.BytesIO(file_content), encoding=encoding)
                    return {'Sheet1': df}
                except:
                    continue
            # If CSV fails, try as Excel
            try:
                return pd.read_excel(io.BytesIO(file_content), sheet_name=None)
            except:
                return None
        else:
            return pd.read_excel(io.BytesIO(file_content), sheet_name=None)
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")
        return None

# Load all uploaded files
dataframes = {}

print("Loading files...\n")
for filename, content in uploaded.items():
    print(f"📊 Loading: {filename}")
    df_dict = read_file(filename, content)

    if df_dict:
        dataframes[filename] = df_dict
        for sheet_name, df in df_dict.items():
            print(f"  ✅ Sheet '{sheet_name}': {len(df)} rows × {len(df.columns)} columns")
    else:
        print(f"  ❌ Failed to load {filename}")

print(f"\n✅ Successfully loaded {len(dataframes)} file(s)")

Loading files...

📊 Loading: Activités par Unité et Saison.xlsx
  ✅ Sheet 'Sheet1': 6 rows × 4 columns
📊 Loading: Activites_Generales.xlsx
  ✅ Sheet 'Sheet1': 4 rows × 4 columns
📊 Loading: american-bsa_merit_badges (1).xlsx
  ✅ Sheet 'Sheet1': 141 rows × 3 columns
📊 Loading: american-bsa_merit_badges.xlsx
  ✅ Sheet 'Sheet1': 141 rows × 3 columns
📊 Loading: american-scout-partners (1).xlsx
  ✅ Sheet 'Sheet1': 38 rows × 2 columns
📊 Loading: american-scout-partners.xlsx
  ✅ Sheet 'Sheet1': 38 rows × 2 columns
📊 Loading: Budgets par Type d'Activité.xlsx
  ✅ Sheet 'Sheet1': 4 rows × 4 columns
📊 Loading: Budgets_et_Finances.xlsx
  ✅ Sheet 'Sheet1': 6 rows × 4 columns
📊 Loading: Camps_Detailles.xlsx
  ✅ Sheet 'Sheet1': 7 rows × 5 columns
📊 Loading: Classeur1.xlsx
  ✅ Sheet 'Feuil1': 5 rows × 3 columns
📊 Loading: Détails des Coûts de Camp (Dinars).xlsx
  ✅ Sheet 'Sheet1': 7 rows × 7 columns
📊 Loading: event-france_final (1).xlsx
  ✅ Sheet 'Sheet1': 232 rows × 9 columns
📊 Loading: event-spain (

# Perform Comprehensive **Analysis**

In [6]:
def analyze_dataframe(df, file_name, sheet_name):
    """Perform comprehensive EDA and quality analysis"""

    analysis = {
        'file': file_name,
        'sheet': sheet_name,
        'dimensions': {
            'rows': len(df),
            'columns': len(df.columns)
        },
        'columns': {},
        'quality': {},
        'visualizations': []
    }

    # Column-level analysis
    for col in df.columns:
        col_info = {
            'dtype': str(df[col].dtype),
            'non_null': int(df[col].notna().sum()),
            'null': int(df[col].isna().sum()),
            'null_pct': float(df[col].isna().sum() / len(df) * 100) if len(df) > 0 else 0,
            'unique': int(df[col].nunique()),
            'unique_pct': float(df[col].nunique() / len(df) * 100) if len(df) > 0 else 0
        }

        # Sample values
        non_null = df[col].dropna()
        if len(non_null) > 0:
            col_info['samples'] = [str(x) for x in non_null.head(3).tolist()]

        # Statistics for numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            stats = df[col].describe()
            col_info['stats'] = {
                'mean': float(stats['mean']) if pd.notna(stats['mean']) else None,
                'std': float(stats['std']) if pd.notna(stats['std']) else None,
                'min': float(stats['min']) if pd.notna(stats['min']) else None,
                'q25': float(stats['25%']) if pd.notna(stats['25%']) else None,
                'median': float(stats['50%']) if pd.notna(stats['50%']) else None,
                'q75': float(stats['75%']) if pd.notna(stats['75%']) else None,
                'max': float(stats['max']) if pd.notna(stats['max']) else None
            }

        analysis['columns'][col] = col_info

    # Quality metrics
    total_cells = len(df) * len(df.columns)
    missing_cells = df.isna().sum().sum()

    analysis['quality'] = {
        'total_cells': int(total_cells),
        'missing_cells': int(missing_cells),
        'completeness_pct': float((total_cells - missing_cells) / total_cells * 100) if total_cells > 0 else 0,
        'duplicates': int(df.duplicated().sum()),
        'duplicate_pct': float(df.duplicated().sum() / len(df) * 100) if len(df) > 0 else 0,
        'empty_columns': [col for col in df.columns if df[col].isna().all()],
        'complete_columns': [col for col in df.columns if df[col].notna().all()]
    }

    return analysis, df

# Analyze all dataframes
all_analyses = []

print("Performing comprehensive analysis...\n")
for file_name, sheets in dataframes.items():
    print(f"📊 Analyzing: {file_name}")
    for sheet_name, df in sheets.items():
        print(f"  Sheet: {sheet_name}")
        analysis, _ = analyze_dataframe(df, file_name, sheet_name)
        all_analyses.append(analysis)
        print(f"  ✅ Quality Score: {analysis['quality']['completeness_pct']:.1f}%")

print(f"\n✅ Analysis complete for {len(all_analyses)} dataset(s)!")

Performing comprehensive analysis...

📊 Analyzing: Activités par Unité et Saison.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Activites_Generales.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-bsa_merit_badges (1).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-bsa_merit_badges.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-scout-partners (1).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-scout-partners.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Budgets par Type d'Activité.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Budgets_et_Finances.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Camps_Detailles.xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Classeur1.xlsx
  Sheet: Feuil1
  ✅ Quality Score: 100.0%
📊 Analyzing: Détails des Coûts de Camp (Dinars).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: event-france_final (1)

Generate Visualizations

In [7]:
def create_visualizations(df, file_name, sheet_name):
    """Create comprehensive visualizations"""

    plot_images = []

    print(f"\n📊 Generating visualizations for {file_name} - {sheet_name}")

    # 1. Missing Data Heatmap
    if df.shape[0] > 0 and df.isnull().any().any():
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='RdYlGn_r', ax=ax)
        ax.set_title(f'Missing Data Pattern - {sheet_name}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Columns')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Missing Data Heatmap', img_buffer))
        plt.close()
        print("  ✅ Missing data heatmap")

    # 2. Missing Data Percentage
    missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    if missing_pct.sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='coral', edgecolor='black')
        ax.set_title(f'Missing Data % by Column - {sheet_name}', fontsize=14, fontweight='bold')
        ax.set_ylabel('Missing %')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Missing Percentage', img_buffer))
        plt.close()
        print("  ✅ Missing percentage chart")

    # 3. Numeric Distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols and len(df) > 0:
        n_cols = min(3, len(numeric_cols))
        fig, axes = plt.subplots(1, n_cols, figsize=(15, 4))
        if n_cols == 1:
            axes = [axes]

        for i, col in enumerate(numeric_cols[:n_cols]):
            data = df[col].dropna()
            if len(data) > 0:
                axes[i].hist(data, bins=30, edgecolor='black', color='steelblue', alpha=0.7)
                axes[i].set_title(f'{col}', fontweight='bold')
                axes[i].set_ylabel('Frequency')

        plt.suptitle('Numeric Distributions', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Numeric Distributions', img_buffer))
        plt.close()
        print("  ✅ Numeric distributions")

    # 4. Categorical Distribution
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    if categorical_cols and len(df) > 0:
        col = categorical_cols[0]
        value_counts = df[col].value_counts().head(10)

        if len(value_counts) > 0:
            fig, ax = plt.subplots(figsize=(12, 6))
            value_counts.plot(kind='barh', ax=ax, color='lightgreen', edgecolor='black')
            ax.set_title(f'{col} - Top 10 Values', fontsize=14, fontweight='bold')
            ax.set_xlabel('Count')
            plt.tight_layout()

            img_buffer = io.BytesIO()
            plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
            img_buffer.seek(0)
            plot_images.append(('Categorical Distribution', img_buffer))
            plt.close()
            print("  ✅ Categorical distribution")

    # 5. Correlation Matrix
    if len(numeric_cols) >= 2:
        fig, ax = plt.subplots(figsize=(10, 8))
        corr_matrix = df[numeric_cols].corr()
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                    square=True, linewidths=1, ax=ax)
        ax.set_title(f'Correlation Matrix - {sheet_name}', fontsize=14, fontweight='bold')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Correlation Matrix', img_buffer))
        plt.close()
        print("  ✅ Correlation matrix")

    return plot_images

# Generate visualizations for all datasets
all_visualizations = {}

for file_name, sheets in dataframes.items():
    all_visualizations[file_name] = {}
    for sheet_name, df in sheets.items():
        viz = create_visualizations(df, file_name, sheet_name)
        all_visualizations[file_name][sheet_name] = viz

print("\n✅ All visualizations generated!")


📊 Generating visualizations for Activités par Unité et Saison.xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for Activites_Generales.xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for american-bsa_merit_badges (1).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution

📊 Generating visualizations for american-bsa_merit_badges.xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution

📊 Generating visualizations for american-scout-partners (1).xlsx - Sheet1
  ✅ Categorical distribution

📊 Generating visualizations for american-scout-partners.xlsx - Sheet1
  ✅ Categorical distribution

📊 Generating visualizations for Budgets par Type d'Activité.xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for Budgets_et_Finances.xlsx - Sheet1
  ✅ Numeric dist

Generate Professional Word Report

In [8]:
def create_word_report(analyses, visualizations, dataframes):
    """Create comprehensive Word document report"""

    doc = Document()

    # Title Page
    title = doc.add_heading('COMPREHENSIVE DATA ANALYSIS REPORT', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    subtitle = doc.add_heading('Exploratory Data Analysis & Data Quality Assessment', level=1)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

    date_para = doc.add_paragraph(f"Generated: {datetime.now().strftime('%B %d, %Y at %H:%M')}")
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()

    # Executive Summary
    doc.add_heading('EXECUTIVE SUMMARY', 1)
    doc.add_paragraph(
        'This comprehensive report presents the Exploratory Data Analysis (EDA) and Data Quality '
        'Assessment for all uploaded datasets. The analysis includes statistical summaries, data '
        'quality metrics, visualizations, and recommendations.'
    )

    doc.add_heading('Analysis Overview', 2)
    doc.add_paragraph(f"Total Files Analyzed: {len(dataframes)}", style='List Bullet')
    doc.add_paragraph(f"Total Datasets: {len(analyses)}", style='List Bullet')
    total_rows = sum(a['dimensions']['rows'] for a in analyses)
    doc.add_paragraph(f"Total Rows: {total_rows:,}", style='List Bullet')

    doc.add_page_break()

    # Individual File Analyses
    for idx, analysis in enumerate(analyses, 1):
        doc.add_heading(f"{idx}. {analysis['file']} - {analysis['sheet']}", 1)

        # Dataset Overview
        doc.add_heading('Dataset Overview', 2)
        doc.add_paragraph(f"Rows: {analysis['dimensions']['rows']:,}", style='List Bullet')
        doc.add_paragraph(f"Columns: {analysis['dimensions']['columns']}", style='List Bullet')

        # Data Quality Metrics
        doc.add_heading('Data Quality Metrics', 2)
        quality = analysis['quality']
        doc.add_paragraph(f"Completeness: {quality['completeness_pct']:.2f}%", style='List Bullet')
        doc.add_paragraph(f"Missing Cells: {quality['missing_cells']:,}", style='List Bullet')
        doc.add_paragraph(f"Duplicate Rows: {quality['duplicates']} ({quality['duplicate_pct']:.2f}%)", style='List Bullet')

        # Quality Rating
        completeness = quality['completeness_pct']
        if completeness >= 95:
            rating = "EXCELLENT ✓"
        elif completeness >= 80:
            rating = "GOOD"
        elif completeness >= 60:
            rating = "FAIR"
        else:
            rating = "NEEDS IMPROVEMENT"

        quality_para = doc.add_paragraph(f"Quality Rating: {rating}")
        quality_para.runs[0].font.bold = True
        quality_para.runs[0].font.size = Pt(14)

        # Column Summary Table
        doc.add_heading('Column Summary', 2)
        table = doc.add_table(rows=1, cols=5)
        table.style = 'Light Grid Accent 1'

        header_cells = table.rows[0].cells
        header_cells[0].text = 'Column'
        header_cells[1].text = 'Type'
        header_cells[2].text = 'Non-Null'
        header_cells[3].text = 'Null %'
        header_cells[4].text = 'Unique'

        for col, col_data in list(analysis['columns'].items())[:10]:
            row_cells = table.add_row().cells
            row_cells[0].text = col
            row_cells[1].text = col_data['dtype']
            row_cells[2].text = f"{col_data['non_null']:,}"
            row_cells[3].text = f"{col_data['null_pct']:.1f}%"
            row_cells[4].text = f"{col_data['unique']:,}"

        if len(analysis['columns']) > 10:
            doc.add_paragraph(f"... and {len(analysis['columns']) - 10} more columns", style='Intense Quote')

        # Visualizations
        doc.add_heading('Visualizations', 2)

        if analysis['file'] in visualizations and analysis['sheet'] in visualizations[analysis['file']]:
            plots = visualizations[analysis['file']][analysis['sheet']]
            for plot_name, plot_buffer in plots:
                doc.add_paragraph(plot_name, style='Intense Quote')
                plot_buffer.seek(0)
                doc.add_picture(plot_buffer, width=Inches(6))
                doc.add_paragraph()

        doc.add_page_break()

    # Overall Summary
    doc.add_heading('OVERALL DATA QUALITY SUMMARY', 1)

    avg_completeness = sum(a['quality']['completeness_pct'] for a in analyses) / len(analyses)
    total_duplicates = sum(a['quality']['duplicates'] for a in analyses)

    doc.add_heading('Key Findings', 2)
    doc.add_paragraph(f"Average Completeness: {avg_completeness:.2f}%", style='List Bullet')
    doc.add_paragraph(f"Total Duplicate Rows: {total_duplicates:,}", style='List Bullet')

    # Recommendations
    doc.add_heading('RECOMMENDATIONS', 1)
    doc.add_paragraph('Address missing data through validation rules', style='List Bullet')
    doc.add_paragraph('Remove duplicate records to ensure data integrity', style='List Bullet')
    doc.add_paragraph('Implement data quality monitoring processes', style='List Bullet')
    doc.add_paragraph('Consider automated data quality checks', style='List Bullet')

    return doc

print("📄 Generating Word document...")
word_doc = create_word_report(all_analyses, all_visualizations, dataframes)
word_filename = f"EDA_and_Quality_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"
word_doc.save(word_filename)
print(f"✅ Word report created: {word_filename}")

📄 Generating Word document...
✅ Word report created: EDA_and_Quality_Report_20260201_1846.docx


Generate Excel Summary

In [9]:
def create_excel_summary(analyses):
    """Create Excel summary of all analyses"""

    summary_data = []

    for analysis in analyses:
        summary_data.append({
            'File': analysis['file'],
            'Sheet': analysis['sheet'],
            'Rows': analysis['dimensions']['rows'],
            'Columns': analysis['dimensions']['columns'],
            'Completeness %': round(analysis['quality']['completeness_pct'], 2),
            'Missing Cells': analysis['quality']['missing_cells'],
            'Duplicate Rows': analysis['quality']['duplicates'],
            'Empty Columns': len(analysis['quality']['empty_columns'])
        })

    df_summary = pd.DataFrame(summary_data)

    excel_filename = f"Analysis_Summary_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary', index=False)

        # Format the sheet
        workbook = writer.book
        worksheet = writer.sheets['Summary']

        from openpyxl.styles import Font, PatternFill, Alignment

        # Header formatting
        header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        header_font = Font(bold=True, color='FFFFFF')

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')

        # Auto-adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            worksheet.column_dimensions[column_letter].width = adjusted_width

    return excel_filename

print("📊 Generating Excel summary...")
excel_filename = create_excel_summary(all_analyses)
print(f"✅ Excel summary created: {excel_filename}")

📊 Generating Excel summary...
✅ Excel summary created: Analysis_Summary_20260201_1846.xlsx


Generate Text Reports

In [10]:
def generate_text_reports(analyses):
    """Generate comprehensive text reports"""

    # EDA Report
    eda_lines = []
    eda_lines.append("=" * 100)
    eda_lines.append("EXPLORATORY DATA ANALYSIS (EDA) REPORT")
    eda_lines.append("=" * 100)
    eda_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    eda_lines.append(f"Total Datasets Analyzed: {len(analyses)}")
    eda_lines.append("")

    for analysis in analyses:
        eda_lines.append("\n" + "=" * 100)
        eda_lines.append(f"FILE: {analysis['file']} - SHEET: {analysis['sheet']}")
        eda_lines.append("=" * 100)

        eda_lines.append("\n1. DATASET DIMENSIONS:")
        eda_lines.append(f"   • Rows: {analysis['dimensions']['rows']:,}")
        eda_lines.append(f"   • Columns: {analysis['dimensions']['columns']}")

        eda_lines.append("\n2. COLUMN ANALYSIS:")
        for col, col_data in analysis['columns'].items():
            eda_lines.append(f"\n   Column: {col}")
            eda_lines.append(f"   ├─ Type: {col_data['dtype']}")
            eda_lines.append(f"   ├─ Non-Null: {col_data['non_null']:,} ({100 - col_data['null_pct']:.1f}%)")
            eda_lines.append(f"   ├─ Null: {col_data['null']:,} ({col_data['null_pct']:.1f}%)")
            eda_lines.append(f"   └─ Unique: {col_data['unique']:,} ({col_data['unique_pct']:.1f}%)")

            if 'stats' in col_data and col_data['stats']:
                stats = col_data['stats']
                eda_lines.append(f"      Statistics: Mean={stats.get('mean', 'N/A')}, Median={stats.get('median', 'N/A')}, Std={stats.get('std', 'N/A')}")

    # Quality Report
    quality_lines = []
    quality_lines.append("=" * 100)
    quality_lines.append("DATA QUALITY REPORT")
    quality_lines.append("=" * 100)
    quality_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    quality_lines.append("")

    for analysis in analyses:
        quality_lines.append("\n" + "=" * 100)
        quality_lines.append(f"FILE: {analysis['file']} - SHEET: {analysis['sheet']}")
        quality_lines.append("=" * 100)

        quality = analysis['quality']

        quality_lines.append("\n1. COMPLETENESS CHECK:")
        completeness = quality['completeness_pct']
        if completeness >= 95:
            quality_lines.append(f"   ✅ EXCELLENT: {completeness:.2f}% complete")
        elif completeness >= 80:
            quality_lines.append(f"   ⚠️  GOOD: {completeness:.2f}% complete")
        elif completeness >= 60:
            quality_lines.append(f"   ⚠️  FAIR: {completeness:.2f}% complete")
        else:
            quality_lines.append(f"   ❌ POOR: {completeness:.2f}% complete")

        quality_lines.append("\n2. DUPLICATE CHECK:")
        if quality['duplicates'] == 0:
            quality_lines.append("   ✅ NO DUPLICATES FOUND")
        else:
            quality_lines.append(f"   ⚠️  {quality['duplicates']:,} duplicate rows ({quality['duplicate_pct']:.2f}%)")

        quality_lines.append("\n3. EMPTY COLUMNS:")
        if quality['empty_columns']:
            quality_lines.append(f"   ⚠️  Columns with ALL nulls: {', '.join(quality['empty_columns'])}")
        else:
            quality_lines.append("   ✅ NO COMPLETELY EMPTY COLUMNS")

    # Save reports
    eda_filename = f"EDA_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
    quality_filename = f"Quality_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"

    with open(eda_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(eda_lines))

    with open(quality_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(quality_lines))

    return eda_filename, quality_filename

print("📝 Generating text reports...")
eda_txt, quality_txt = generate_text_reports(all_analyses)
print(f"✅ EDA text report: {eda_txt}")
print(f"✅ Quality text report: {quality_txt}")

📝 Generating text reports...
✅ EDA text report: EDA_Report_20260201_1847.txt
✅ Quality text report: Quality_Report_20260201_1847.txt


Download All Reports

In [11]:
print("📥 Downloading all reports...\n")

# Download Word report
print(f"📄 Downloading: {word_filename}")
files.download(word_filename)

# Download Excel summary
print(f"📊 Downloading: {excel_filename}")
files.download(excel_filename)

# Download text reports
print(f"📝 Downloading: {eda_txt}")
files.download(eda_txt)

print(f"📝 Downloading: {quality_txt}")
files.download(quality_txt)

print("\n✅ All reports downloaded successfully!")

📥 Downloading all reports...

📄 Downloading: EDA_and_Quality_Report_20260201_1846.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📊 Downloading: Analysis_Summary_20260201_1846.xlsx


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

📝 Downloading: Quality_Report_20260201_1847.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All reports downloaded successfully!


Analysis Summary

In [12]:
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)

for analysis in all_analyses:
    print(f"\n📊 {analysis['file']} - {analysis['sheet']}")
    print(f"   Dimensions: {analysis['dimensions']['rows']:,} rows × {analysis['dimensions']['columns']} columns")
    print(f"   Completeness: {analysis['quality']['completeness_pct']:.2f}%")
    print(f"   Duplicates: {analysis['quality']['duplicates']:,}")

    # Quality rating
    completeness = analysis['quality']['completeness_pct']
    if completeness >= 95:
        rating = "✅ EXCELLENT"
    elif completeness >= 80:
        rating = "⚠️  GOOD"
    elif completeness >= 60:
        rating = "⚠️  FAIR"
    else:
        rating = "❌ NEEDS IMPROVEMENT"
    print(f"   Quality: {rating}")

print("\n" + "="*80)
print(f"Total Datasets: {len(all_analyses)}")
avg_quality = sum(a['quality']['completeness_pct'] for a in all_analyses) / len(all_analyses)
print(f"Average Quality: {avg_quality:.2f}%")
print("="*80)


ANALYSIS SUMMARY

📊 Activités par Unité et Saison.xlsx - Sheet1
   Dimensions: 6 rows × 4 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 Activites_Generales.xlsx - Sheet1
   Dimensions: 4 rows × 4 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-bsa_merit_badges (1).xlsx - Sheet1
   Dimensions: 141 rows × 3 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-bsa_merit_badges.xlsx - Sheet1
   Dimensions: 141 rows × 3 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-scout-partners (1).xlsx - Sheet1
   Dimensions: 38 rows × 2 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-scout-partners.xlsx - Sheet1
   Dimensions: 38 rows × 2 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 Budgets par Type d'Activité.xlsx - Sheet1
   Dimensions: 4 rows × 4 columns
   Completeness: 100.00%
   Duplicate